<a href="https://colab.research.google.com/github/fickas/crab_project/blob/main/notebooks/saltmarsh_crab_generate_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Synthetic data generation

#Run once

In [1]:
# Cell 1 — repo + dependencies
!git clone https://github.com/fickas/crab_project.git /content/marsh-crab 2>/dev/null || \
 (cd /content/marsh-crab && git pull)
!pip install -r /content/marsh-crab/requirements.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.1 MB/s eta 0:00:00


#Wait until marsh-crab shows up locally

In [2]:
# Cell 2 — Python setup
import sys
sys.path.insert(0, '/content/marsh-crab')
import marsh_utils as mu

##Run on changes

In [3]:
# Run this cell whenever you edit the repo
!cd /content/marsh-crab && git pull
import importlib
importlib.reload(mu)

Already up to date.


<module 'marsh_utils' from '/content/marsh-crab/marsh_utils.py'>

#

In [4]:
"""
Generate synthetic Wellfleet-like marsh data for pipeline testing.

Run this in a Colab cell to produce two synthetic datasets on your Drive:
  - {BASE}/synthetic_test/wellfleet_high_res_synthetic/ — 30m × 30m
        ms_5band.tif       (2 cm, raw multispectral B/G/R/RE/NIR)
        pan.tif            (1 cm, panchromatic)
        pansharp_5band.tif (1 cm, pansharpened B/G/R/RE/NIR)
        dem_5m.tif         (50 cm)
        crab_polygons.shp  (the "hand-labels" you'd draw in QGIS)
  - {BASE}/synthetic_test/wellfleet_normal_synthetic/  — 150m × 150m
        ms_5band.tif       (4 cm, raw multispectral)
        dem_5m.tif         (50 cm)
        other_handlabels.shp ('other'-class polygons)

NDVI / NDRE are NOT precomputed — `ensure_indices()` in your pipeline will
compute and cache them on first read of pansharp_5band.tif / ms_5band.tif.

WORKFLOW
========
1. Run this once to produce the synthetic data.
2. Model 1 notebook: point `paths` at wellfleet_high_res_synthetic/.
   Set Config.BLOCK_SIZE_M = 10 (default 100m doesn't fit a 30m extent).
3. Run Model 1 inference → produces model1_polygons.shp.
4. Model 2 notebook: point `paths` at wellfleet_normal_synthetic/, plus the
   Model 1 output polygons + other_handlabels.shp. Set Config.BLOCK_SIZE_M = 30.
"""
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import LineString, Polygon, Point, box, MultiPolygon
from shapely.ops import unary_union
from scipy import ndimage
import shutil

# ============================================================================
# CUSTOMIZE THIS
# ============================================================================
BASE = '/content/drive/MyDrive/salt_marsh/crab_project/WEL'

# ============================================================================
# Config
# ============================================================================

# Model 2 extent (full synthetic marsh)
FULL_EXTENT_M = 150.0
CX, CY = 410000.0, 4642500.0     # approximate Wellfleet UTM 19N coords

# Model 1 extent (high-res strip; a window inside the M2 area)
M1_EXTENT_M = 60.0          # 4x more area → 4x more polygons per class
M1_OFFSET = (-15, 5)        # Adjusted to keep M1 inside the full marsh


def main():
    rng = np.random.default_rng(mu.SEED)

    # Full-marsh bounds (Model 2 area)
    full_bounds = box(CX - FULL_EXTENT_M/2, CY - FULL_EXTENT_M/2,
                     CX + FULL_EXTENT_M/2, CY + FULL_EXTENT_M/2)

    print("Generating underlying marsh geometry...")
    geom_full = mu.generate_marsh_geometry(full_bounds, rng)

    print("Assigning polygon labels...")
    polygons_full = mu.assign_polygon_labels(geom_full, rng)
    print(f"  Total polygons: {len(polygons_full)}")
    name_lookup = {v: k for k, v in mu.CLASSES.items()}
    for class_id, count in polygons_full['Class'].value_counts().sort_index().items():
        print(f"    {class_id} ({name_lookup[class_id]}): {count}")

    # ---- Model 1 dataset (1cm GSD, small strip) ----
    m1_cx = CX + M1_OFFSET[0]
    m1_cy = CY + M1_OFFSET[1]
    m1_bounds = box(m1_cx - M1_EXTENT_M/2, m1_cy - M1_EXTENT_M/2,
                    m1_cx + M1_EXTENT_M/2, m1_cy + M1_EXTENT_M/2)
    geom_m1 = {k: (v.intersection(m1_bounds) if hasattr(v, 'intersection')
                   else [g.intersection(m1_bounds) for g in v])
               for k, v in geom_full.items()}
    polygons_m1 = gpd.clip(polygons_full, m1_bounds)
    polygons_m1 = polygons_m1[polygons_m1.area > 0.05].reset_index(drop=True)
    polygons_m1 = mu.ensure_all_classes_in_m1(polygons_m1, m1_bounds, geom_m1, rng)

    SYNTHETIC_ROOT = f'{BASE}/synthetic_test'
    m1_dir = f'{SYNTHETIC_ROOT}/wellfleet_high_res_synthetic'

    # Nuke any stale data from a previous run (derived bands, old DEMs, etc.)
    if os.path.exists(m1_dir):
        print(f"  Removing existing {m1_dir}...")
        shutil.rmtree(m1_dir)
    os.makedirs(m1_dir, exist_ok=True)

    # Model 1: pan at 1cm, pansharp_5band at 1cm, ms_5band at 2cm
    mu.generate_dataset(m1_dir, m1_bounds, geom_m1, polygons_m1,
                     'model1_high_res',
                     pan_gsd_m=0.01, ms_gsd_m=0.02,
                     seed_offset=1)
    polygons_m1.to_file(f'{m1_dir}/crab_polygons.shp')
    print(f"  Wrote {len(polygons_m1)} polygons to crab_polygons.shp")

    # ---- Model 2 dataset (4cm GSD, full marsh) ----
    # Model 2 covers the entire synthetic marsh
    m2_bounds   = full_bounds       # the same 150 m × 150 m extent
    geom_m2     = geom_full         # same geometry — channels, banks, trees, hummock, ponds
    polygons_m2 = polygons_full     # all labeled polygons across the full marsh
    m2_dir = f'{SYNTHETIC_ROOT}/wellfleet_normal_synthetic'

        # Nuke any stale data from a previous run (derived bands, old DEMs, etc.)
    if os.path.exists(m2_dir):
        print(f"  Removing existing {m2_dir}...")
        shutil.rmtree(m2_dir)
    os.makedirs(m2_dir, exist_ok=True)

    # If your Model 2 also needs pan + pansharp at 2cm, set pan_gsd_m=0.02 here too.
    mu.generate_dataset(m2_dir, m2_bounds, geom_m2, polygons_m2,
                 'model2_normal',
                 pan_gsd_m=0.04,        # NEW — high-altitude pan at 4 cm
                 ms_gsd_m=0.08,         # changed from 0.04 — MS at 8 cm
                 seed_offset=2)

# Model 2 'other' hand-labels — generated to mimic what a human would draw in QGIS
    other_polys = mu.generate_other_handlabels(geom_full, full_bounds, rng,
                                                n_marsh_interior=10,
                                                n_mud_patches=5,
                                                n_wrack=3)
    other_polys.to_file(f'{m2_dir}/other_handlabels.shp')

    print(f"  Wrote {len(other_polys)} 'other'-class polygons to other_handlabels.shp")
    print(f"\nDone. Synthetic data ready at:")
    print(f"  {m1_dir}")
    print(f"  {m2_dir}")
    print(f"\nIn your Model 1 notebook, set:")
    print(f"  base = '{BASE}'")
    print(f"  paths['pan_orthomosaic'] = f'{{base}}/synthetic_test/wellfleet_high_res_synthetic/pan.tif'")
    print(f"  ... etc.")
    print(f"  AND set Config.BLOCK_SIZE_M = 10  (smaller blocks for the small test extent)")


main()

Generating underlying marsh geometry...
Assigning polygon labels...
  Total polygons: 153
    1 (other): 6
    2 (healthy_bank): 52
    3 (eroding_non_crab): 26
    4 (crab_edge): 35
    5 (crab_platform): 21
    6 (collapsed): 13
  Removing existing /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic...

Generating dataset: model1_high_res
  Output dir: /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_high_res_synthetic
  Raster: 6000×6000 pixels at 1.0cm GSD (36.0M pixels)
  Compositing spectral bands at 1.0cm GSD...
  Writing pan.tif (1.0cm)...
  Writing pansharp_5band.tif (1.0cm)...
  Writing ms_5band.tif (2.0cm)...
  Writing dem_5m.tif at imagery GSD (1.0cm)...
  Wrote 99 polygons to crab_polygons.shp
  Removing existing /content/drive/MyDrive/salt_marsh/crab_project/WEL/synthetic_test/wellfleet_normal_synthetic...

Generating dataset: model2_normal
  Output dir: /content/drive/MyDrive/salt_marsh/crab_project/WE